# Part A — Single-Country Optimal Capacity Mix (Denmark)

1. **Section 1 (Optimisation)** — run once, saves results to `plots_part_a/`
2. **Section 2 (Plotting)** — loads from CSV, re-run freely without the solver

---
## Section 1 — Optimisation
*Run these cells once. Skip after CSVs exist.*

In [ ]:
import os
import pandas as pd
import pypsa

df_elec = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0)
df_elec.index = pd.to_datetime(df_elec.index)

df_onshorewind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0)
df_onshorewind.index = pd.to_datetime(df_onshorewind.index)

df_offshorewind = pd.read_csv('data/offshore_wind_1979-2017.csv', sep=';', index_col=0)
df_offshorewind.index = pd.to_datetime(df_offshorewind.index)

df_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0)
df_solar.index = pd.to_datetime(df_solar.index)

country = 'DNK'

def annuity(n, r):
    return r / (1. - 1. / (1. + r) ** n) if r > 0 else 1 / n

In [ ]:
network = pypsa.Network()
hours_in_2015 = pd.date_range('2015-01-01 00:00Z', '2015-12-31 23:00Z', freq='h')
network.set_snapshots(hours_in_2015.values)
network.add("Bus", "electricity bus")
network.add("Load", "load", bus="electricity bus", p_set=df_elec[country].values)

network.add("Carrier", "gas", co2_emissions=0.19)
network.add("Carrier", "onshorewind")
network.add("Carrier", "offshorewind")
network.add("Carrier", "solar")

CF_wind = df_onshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
network.add("Generator", "Onshore wind", bus="electricity bus",
            p_nom_extendable=True, carrier="onshorewind",
            capital_cost=annuity(27, 0.07) * 1118775, marginal_cost=0,
            p_max_pu=CF_wind.values)

CF_wind_off = df_offshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
network.add("Generator", "Offshore wind", bus="electricity bus",
            p_nom_extendable=True, carrier="offshorewind",
            capital_cost=annuity(27, 0.07) * 2115944, marginal_cost=0,
            p_max_pu=CF_wind_off.values)

CF_solar = df_solar[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
network.add("Generator", "Solar", bus="electricity bus",
            p_nom_extendable=True, carrier="solar",
            capital_cost=annuity(25, 0.07) * 450000, marginal_cost=0,
            p_max_pu=CF_solar.values)

network.add("Generator", "OCGT", bus="electricity bus",
            p_nom_extendable=True, carrier="gas",
            capital_cost=annuity(25, 0.07) * 453960, marginal_cost=30 / 0.41)

network.add("Generator", "CCGT", bus="electricity bus",
            p_nom_extendable=True, carrier="gas",
            capital_cost=annuity(25, 0.07) * 880000, marginal_cost=30 / 0.56)

network.optimize(solver_name='gurobi', solver_options={"OutputFlag": 0, "LogToConsole": 0})

print(f"Total system cost: {network.objective / 1e6:.2f} M\u20ac/yr")
print(f"Average cost:      {network.objective / network.loads_t.p.sum().sum():.2f} \u20ac/MWh")
print("\nOptimal capacities [GW]:")
print(network.generators.p_nom_opt.div(1e3))

In [ ]:
os.makedirs('plots_part_a', exist_ok=True)

network.generators.p_nom_opt.to_frame('p_nom_opt').to_csv('plots_part_a/generators_capacity.csv')
network.generators_t.p.to_csv('plots_part_a/dispatch.csv')
network.loads_t.p.to_csv('plots_part_a/load.csv')
print("Saved CSVs to plots_part_a/")

---
## Section 2 — Plotting
*Start here after CSVs exist. No solver needed.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cap     = pd.read_csv('plots_part_a/generators_capacity.csv', index_col=0)['p_nom_opt']
dispatch = pd.read_csv('plots_part_a/dispatch.csv', index_col=0, parse_dates=True)
load     = pd.read_csv('plots_part_a/load.csv', index_col=0, parse_dates=True)['load']

LABELS      = ['Onshore wind', 'Offshore wind', 'Solar PV', 'Gas (OCGT)', 'Gas (CCGT)']
MODEL_NAMES = ['Onshore wind', 'Offshore wind', 'Solar',    'OCGT',       'CCGT']
COLORS      = ['blue', 'dodgerblue', 'orange', 'crimson', 'darkviolet']

# Drop zero-capacity technologies
active = [(l, n, c) for l, n, c in zip(LABELS, MODEL_NAMES, COLORS) if cap.get(n, 0) > 0]
labels_a, model_names_a, colors_a = zip(*active)

print("Loaded results for", len(dispatch), "hours")
cap

In [ ]:
cap_sizes = [cap[n] for n in model_names_a]
plt.figure(figsize=(6, 5), dpi=150)
plt.pie(cap_sizes, colors=colors_a,
        labels=[f'{l}\n{s/1e3:.1f} GW' for l, s in zip(labels_a, cap_sizes)],
        wedgeprops={'linewidth': 0})
plt.axis('equal')
plt.title('Installed capacity mix', y=1.05, fontweight='bold')
plt.tight_layout()
plt.savefig('plots_part_a/1a_installed_capacity_mix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), dpi=150)
plt.plot(load.iloc[0:168].values, color='grey', label='Demand', lw=3)
for n, l, c in zip(model_names_a, labels_a, colors_a):
    plt.plot(dispatch[n].iloc[0:168].values, color=c, label=l)
plt.ylabel('Generation [MWh/h]')
plt.title('Electricity generation — first week of January 2015')
plt.xlabel('Hour of week')
plt.legend(fancybox=True, shadow=True, loc='best')
plt.tight_layout()
plt.savefig('plots_part_a/1a_timeseries_winter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), dpi=150)
plt.plot(load.iloc[4344:4512].values, color='grey', label='Demand', lw=3)
for n, l, c in zip(model_names_a, labels_a, colors_a):
    plt.plot(dispatch[n].iloc[4344:4512].values, color=c, label=l)
plt.ylabel('Generation [MWh/h]')
plt.title('Electricity generation — first week of July 2015')
plt.xlabel('Hour of week')
plt.legend(fancybox=True, shadow=True, loc='best')
plt.tight_layout()
plt.savefig('plots_part_a/1a_timeseries_summer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
sizes = [dispatch[n].sum() for n in model_names_a]
percentages = [s / sum(sizes) * 100 for s in sizes]
plt.figure(figsize=(6, 5), dpi=150)
plt.pie(sizes, colors=colors_a,
        labels=[f'{l}\n{p:.1f}%' for l, p in zip(labels_a, percentages)],
        wedgeprops={'linewidth': 0})
plt.axis('equal')
plt.title('Annual electricity generation mix', y=1.05, fontweight='bold')
plt.tight_layout()
plt.savefig('plots_part_a/1a_annual_generation_mix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), dpi=150)
for n, l, c in zip(model_names_a, labels_a, colors_a):
    plt.plot(dispatch[n].sort_values(ascending=False).reset_index(drop=True).values,
             label=l, color=c, lw=2.5)
plt.ylabel('Generation [MWh/h]')
plt.xlabel('Hours')
plt.title('Duration curve of generation')
plt.legend(fancybox=True, shadow=True, loc='best')
plt.tight_layout()
plt.savefig('plots_part_a/1a_duration_curve.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 5), dpi=150)
for n, l, c in zip(model_names_a, labels_a, colors_a):
    cf = dispatch[n].sort_values(ascending=False).reset_index(drop=True) / cap[n]
    plt.plot(cf.values, label=l, color=c, lw=2.5)
plt.ylabel('Capacity factor [-]')
plt.xlabel('Hours')
plt.title('Normalised duration curve (capacity factors)')
plt.legend(fancybox=True, shadow=True, loc='best')
plt.tight_layout()
plt.savefig('plots_part_a/1a_duration_curve_CFs.png', dpi=300, bbox_inches='tight')
plt.show()